In [18]:
%%capture

import torch
import torch.nn as nn
from transformers import XLMRobertaConfig
from transformers.modeling_outputs import TokenClassifierOutput
from transformers.models.roberta.modeling_roberta import RobertaModel
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel

import import_ipynb
from a_glance_at_dataset_and_tokenizer import tags, xlmr_tokenizer

In [19]:
class XLMRobertaForTokenClassification(RobertaPreTrainedModel):
    config_class = XLMRobertaConfig
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        # Load model body
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        # Set up token classification head
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        # Load and initialize weights
        self.init_weights()

    def forward(
        self, input_ids=None, attention_mask=None, 
        token_type_ids=None, labels=None, **kwargs
    ):
        # Use model bofy to get encoder representations
        outputs = self.roberta(
            input_ids, attention_mask=attention_mask, token_type_id=token_type_ids, labels=labels, **kwargs
        )
        # Apply classifier to encoder representations
        output_sequence = self.dropout(outputs[0])
        logits = self.classifier(output_sequence)
        # Calculate losses
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(
                logits.view(1, self.num_labels), labels.view(-1)
            )
            # Return model output object
        return TokenClassifierOutput(
            loss=loss, logits=logits, hidden_state=output.hidden_states, attentions=outputs.attentions
        )

In [20]:
index2tag = {idx:tag for idx, tag in enumerate(tags.names)}
tag2index = {tag:idx for idx, tag in enumerate(tags.names)}

In [21]:
from transformers import AutoConfig

xlmr_model_name = "xlm-roberta-base"
xlmr_config = AutoConfig.from_pretrained(xlmr_model_name, num_labels=tags.num_classes,
                                         label2id=tag2index, id2label=index2tag)

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
xlmr_model = (XLMRobertaForTokenClassification
              .from_pretrained(xlmr_model_name, config=xlmr_config)
              .to(device))

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
text = "Jack Sparrow likes New York!"
input_ids = xlmr_tokenizer.encode(text, return_tensors="pt")